### **IMPORTO LIBRERIE**

In [ ]:
import pandas as pd
import numpy as np

### **IMPORTO I DATI**

In [ ]:
#Importo i fogli excel come dataframe per l'analisi
df_software = pd.read_excel("datas/raw/dataset_software.xlsx", sheet_name="Risultati")
df_hardware = pd.read_excel("datas/raw/dataset_hardware.xlsx", sheet_name="Risultati")

#Stampo dimensione campione iniziale
print("Dimensione campione iniziale: " + str(len(df_software) + len(df_hardware)) + " osservazioni.")

### **PULISCO I DATI**

In [ ]:
variabili_anni = {
        "ROE": [
            "Redditività del capitale proprio (ROE) - Netto 2024",
            "Redditività del capitale proprio (ROE) - Netto 2023",
            "Redditività del capitale proprio (ROE) - Netto 2022",
            "Redditività del capitale proprio (ROE) - Netto 2021",
            "Redditività del capitale proprio (ROE) - Netto 2020",
        ],
        "ROA": [
            "Redditività del totale Attivo (ROA) - Netto 2024",
            "Redditività del totale Attivo (ROA) - Netto 2023",
            "Redditività del totale Attivo (ROA) - Netto 2022",
            "Redditività del totale Attivo (ROA) - Netto 2021",
            "Redditività del totale Attivo (ROA) - Netto 2020",
        ],

        "Indice di liquidita": [
            "Indice di liquidità 2024",
            "Indice di liquidità 2023",
            "Indice di liquidità 2022",
            "Indice di liquidità 2021",
            "Indice di liquidità 2020",
        ],

        "Debt/Equity": [
            "Debt/Equity 2024",
            "Debt/Equity 2023",
            "Debt/Equity 2022",
            "Debt/Equity 2021",
            "Debt/Equity 2020",
        ],

        "Totale Attivo": [
            "Totale Attivo migl USD 2024",
            "Totale Attivo migl USD 2023",
            "Totale Attivo migl USD 2022",
            "Totale Attivo migl USD 2021",
            "Totale Attivo migl USD 2020",
        ],
    }

Rimuovo le righe che hanno valori mancanti

In [ ]:
def pulizia_valori_mancanti(df):
    df = df.copy()

    valori_da_rimuovere = ["n.d.", "n.s.", "nan", "none", ""]

    #converto le colonne in testo cosi da poter lavorare sui valori
    df_testo = (
        df.astype("string")
        .apply(lambda colonna: colonna.str.strip().str.lower())
    )

    maschera_da_tenere = pd.Series(True, index=df.index)
    variabili_con_minimo_tre_anni = [
        "ROE",
        "ROA",
        "Indice di liquidita",
        "Debt/Equity",
    ]

    #controllo che gli anni validi (per variabile) siano almeno 3
    for variabile in variabili_con_minimo_tre_anni:
        colonne = variabili_anni[variabile]
        valori_validi = ~df_testo[colonne].isin(valori_da_rimuovere)
        numero_anni_validi = valori_validi.sum(axis=1)
        maschera_da_tenere = maschera_da_tenere & (numero_anni_validi >= 3)

    df = df[maschera_da_tenere].copy()

    for colonna in df.columns:
        if df[colonna].dtype == "object":
            serie = df[colonna].astype("string").str.strip()
            df[colonna] = serie.mask(
                serie.str.lower().isin(valori_da_rimuovere),
                np.nan
            )

    return df

df_software = pulizia_valori_mancanti(df_software)
df_hardware = pulizia_valori_mancanti(df_hardware)

#Stampo la dimensione del campione statistico a disposizione dopo la pulizia
print("Dimensione campione software pulito: " + str(len(df_software)) + " osservazioni")
print("Dimensione campione hardware pulito: " + str(len(df_hardware)) + " osservazioni")
print("Dimensione campione totale pulito: " + str(len(df_software) + len(df_hardware)) + " osservazioni.")

Converto correttamente le colonne numeriche

In [ ]:
colonne_numeriche = [
    "Numero dipendenti Ultimo anno disp.",

    "Redditività del capitale proprio (ROE) - Netto 2024",
    "Redditività del capitale proprio (ROE) - Netto 2023",
    "Redditività del capitale proprio (ROE) - Netto 2022",
    "Redditività del capitale proprio (ROE) - Netto 2021",
    "Redditività del capitale proprio (ROE) - Netto 2020",

    "Redditività del totale Attivo (ROA) - Netto 2024",
    "Redditività del totale Attivo (ROA) - Netto 2023",
    "Redditività del totale Attivo (ROA) - Netto 2022",
    "Redditività del totale Attivo (ROA) - Netto 2021",
    "Redditività del totale Attivo (ROA) - Netto 2020",

    "Indice di liquidità 2024",
    "Indice di liquidità 2023",
    "Indice di liquidità 2022",
    "Indice di liquidità 2021",
    "Indice di liquidità 2020",

    "Debt/Equity 2024",
    "Debt/Equity 2023",
    "Debt/Equity 2022",
    "Debt/Equity 2021",
    "Debt/Equity 2020",

    "Totale Attivo migl USD 2024",
    "Totale Attivo migl USD 2023",
    "Totale Attivo migl USD 2022",
    "Totale Attivo migl USD 2021",
    "Totale Attivo migl USD 2020",
]

#Creo una funzione per convertire le colonne numeriche correttamente.
def converti_colonne_numeriche(df, colonne):
  df = df.copy()
  valori_mancanti = ["n.d.", "n.s.", "", "nan", "none"]

  for colonna in colonne:
    serie = df[colonna].astype("string").str.strip()
    serie = serie.mask(serie.str.lower().isin(valori_mancanti), pd.NA)

    usa_virgola_decimale = serie.str.contains(",", na=False)

    serie.loc[usa_virgola_decimale] = (
        serie.loc[usa_virgola_decimale]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    df[colonna] = pd.to_numeric(serie, errors="coerce")

  return df


df_hardware = converti_colonne_numeriche(df_hardware, colonne_numeriche)
df_software = converti_colonne_numeriche(df_software, colonne_numeriche)

Converto correttamente le colonne con date

In [ ]:
colonne_date = [
    "Data chiusura 2024",
    "Data chiusura 2023",
    "Data chiusura 2022",
    "Data chiusura 2021",
    "Data chiusura 2020",
    "Data di costituzione",
]

def converti_colonne_date(df, colonne):
    df = df.copy()
    for colonna in colonne:
        serie = df[colonna]

        if pd.api.types.is_datetime64_any_dtype(serie):
            df[colonna] = pd.to_datetime(serie, errors="coerce")
            continue

        date_convertite = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
        valori_numerici = pd.to_numeric(serie, errors="coerce")
        maschera_seriali_excel = valori_numerici.notna()

        date_convertite.loc[maschera_seriali_excel] = pd.to_datetime(
            valori_numerici.loc[maschera_seriali_excel],
            unit="D",
            origin="1899-12-30",
            errors="coerce"
        )

        serie_testo = serie.astype("string").str.strip()
        maschera_testo = ~maschera_seriali_excel & serie_testo.notna()

        date_convertite.loc[maschera_testo] = pd.to_datetime(
            serie_testo.loc[maschera_testo],
            format="%Y-%m-%d %H:%M:%S",
            errors="coerce"
        )

        maschera_solo_data = maschera_testo & date_convertite.isna()
        date_convertite.loc[maschera_solo_data] = pd.to_datetime(
            serie_testo.loc[maschera_solo_data],
            format="%Y-%m-%d",
            errors="coerce"
        )

        df[colonna] = date_convertite

    return df

df_hardware = converti_colonne_date(df_hardware, colonne_date)
df_software = converti_colonne_date(df_software, colonne_date)

Trasformo i "Total assets" in valori logaritmici per normalizzare le distribuzione

In [ ]:
colonne_attivo = [
    "Totale Attivo migl USD 2024",
    "Totale Attivo migl USD 2023",
    "Totale Attivo migl USD 2022",
    "Totale Attivo migl USD 2021",
    "Totale Attivo migl USD 2020",
]


def log_attivo(df, colonne_attivo):
    df = df.copy()

    attivo = df[colonne_attivo].astype(float)

    # Tengo solo valori strettamente positivi
    attivo_positivo = attivo.where(attivo > 0)

    # Calcolo il log naturale solo sui valori positivi
    log_attivo = np.log(attivo_positivo)

    # Calcolo la media dei log per ogni azienda
    df["log_attivo_medio"] = log_attivo.mean(axis=1, skipna=True)

    return df


df_hardware = log_attivo(df_hardware, colonne_attivo)
df_software = log_attivo(df_software, colonne_attivo)

Rimuovo le aziende con Debt/Equity minore o uguale a zero. Queste influiscono negativamente portando i ROE a valori anomali

In [ ]:
def pulizia_debt_equity_positivo(df, colonne_debt_equity):
    df = df.copy()

    debt_equity = df[colonne_debt_equity]

    # Rimuovo le aziende con almeno un valore osservato di Debt/Equity minore o uguale a zero.
    maschera_da_tenere = (debt_equity > 0) | debt_equity.isna()

    return df[maschera_da_tenere.all(axis=1)].copy()


df_hardware = pulizia_debt_equity_positivo(df_hardware, variabili_anni["Debt/Equity"])
df_software = pulizia_debt_equity_positivo(df_software, variabili_anni["Debt/Equity"])

print("Dimensione campione software dopo pulizia Debt/Equity positivo: " + str(len(df_software)) + " osservazioni")
print("Dimensione campione hardware dopo pulizia Debt/Equity positivo: " + str(len(df_hardware)) + " osservazioni")
print("Dimensione campione totale dopo pulizia Debt/Equity positivo: " + str(len(df_software) + len(df_hardware)) + " osservazioni.")

# Salvo una copia prima della rimozione outlier per poter confrontare diversi criteri
df_hardware_pre_outlier = df_hardware.copy()
df_software_pre_outlier = df_software.copy()

Calcolo i valori medi di ROE e ROA

In [ ]:
def calcola_roe_medio(df, colonne_roe, colonna_roe="roe_medio"):
    df = df.copy()
    roe = df[colonne_roe].apply(pd.to_numeric, errors="coerce")
    df[colonna_roe] = roe.mean(axis=1, skipna=True)
    return df


def calcola_roa_medio(df, colonne_roa, colonna_roa="roa_medio"):
    df = df.copy()
    roa = df[colonne_roa].apply(pd.to_numeric, errors="coerce")
    df[colonna_roa] = roa.mean(axis=1, skipna=True)
    return df


df_software = calcola_roe_medio(df_software, variabili_anni["ROE"])
df_hardware = calcola_roe_medio(df_hardware, variabili_anni["ROE"])
df_software = calcola_roa_medio(df_software, variabili_anni["ROA"])
df_hardware = calcola_roa_medio(df_hardware, variabili_anni["ROA"])

Pulizia outlier: winsorizzazione al 1° e 99° percentile sulle medie delle variabili analizzate

In [ ]:
def winsorizza_outlier_su_medie(
    df,
    colonne_roe,
    colonne_roa,
    colonne_debt_equity,
    colonne_liquidita):

    df = df.copy()

    # Calcolo delle medie per azienda
    df = calcola_roe_medio(df, colonne_roe)
    df = calcola_roa_medio(df, colonne_roa)
    df["debt_equity_medio"] = df[colonne_debt_equity].mean(axis=1, skipna=True)
    df["liquidita_media"] = df[colonne_liquidita].mean(axis=1, skipna=True)

    colonne_medie = [
        "roe_medio",
        "roa_medio",
        "debt_equity_medio",
        "liquidita_media"
    ]

    for colonna in colonne_medie:
        limite_inferiore = df[colonna].quantile(0.01)
        limite_superiore = df[colonna].quantile(0.99)

        df[colonna] = df[colonna].clip(
            lower=limite_inferiore,
            upper=limite_superiore
        )

    return df

df_hardware = winsorizza_outlier_su_medie(df_hardware_pre_outlier, variabili_anni["ROE"], variabili_anni["ROA"], variabili_anni["Debt/Equity"], variabili_anni["Indice di liquidita"])
df_software = winsorizza_outlier_su_medie(df_software_pre_outlier, variabili_anni["ROE"], variabili_anni["ROA"], variabili_anni["Debt/Equity"], variabili_anni["Indice di liquidita"])

print("Dimensione campione software con winsorizzazione sulle medie: " + str(len(df_software)) + " osservazioni")
print("Dimensione campione hardware con winsorizzazione sulle medie: " + str(len(df_hardware)) + " osservazioni")
print("Dimensione campione totale con winsorizzazione sulle medie: " + str(len(df_software) + len(df_hardware)) + " osservazioni.")

Pulizia outlier per Test di Resistenza: IRQ sulle medie delle variabili

In [ ]:
def rimuovi_outlier_iqr_su_medie(
    df,
    colonne_roe,
    colonne_roa,
    colonne_debt_equity,
    colonne_liquidita):

    df = df.copy()

    # Calcolo delle medie per azienda
    df = calcola_roe_medio(df, colonne_roe)
    df = calcola_roa_medio(df, colonne_roa)
    df["debt_equity_medio"] = df[colonne_debt_equity].mean(axis=1, skipna=True)
    df["liquidita_media"] = df[colonne_liquidita].mean(axis=1, skipna=True)

    colonne_medie = [
        "roe_medio",
        "roa_medio",
        "debt_equity_medio",
        "liquidita_media"
    ]

    maschera_da_tenere = pd.Series(True, index=df.index)

    for colonna in colonne_medie:
        q1 = df[colonna].quantile(0.25)
        q3 = df[colonna].quantile(0.75)
        iqr = q3 - q1

        limite_inferiore = q1 - 1.5 * iqr
        limite_superiore = q3 + 1.5 * iqr

        maschera_colonna = (
            (df[colonna] >= limite_inferiore) &
            (df[colonna] <= limite_superiore)
        )

        maschera_da_tenere = maschera_da_tenere & maschera_colonna

    df = df[maschera_da_tenere].copy()

    return df

df_hardware_test = rimuovi_outlier_iqr_su_medie(df_hardware, variabili_anni["ROE"], variabili_anni["ROA"], variabili_anni["Debt/Equity"], variabili_anni["Indice di liquidita"])
df_software_test = rimuovi_outlier_iqr_su_medie(df_software, variabili_anni["ROE"], variabili_anni["ROA"], variabili_anni["Debt/Equity"], variabili_anni["Indice di liquidita"])

print("Dimensione campione software pulito: " + str(len(df_software_test)) + " osservazioni")
print("Dimensione campione hardware pulito: " + str(len(df_hardware_test)) + " osservazioni")
print("Dimensione campione totale pulito: " + str(len(df_software_test) + len(df_hardware_test)) + " osservazioni.")

### **SALVO I DATASET PULITI**

In [ ]:
df_hardware.to_csv("datas/processed/dataset_hardware.csv", index=False)
df_software.to_csv("datas/processed/dataset_software.csv", index=False)

df_hardware_test.to_csv("datas/processed/dataset_hardware_test.csv", index=False)
df_software_test.to_csv("datas/processed/dataset_software_test.csv", index=False)